In [1]:
%pip install "autogluon==1.6.3" "joblib==1.6.0" "numpy==2.4.6" "pandas==2.3.3" "scikit-learn==1.9.1" "scipy==1.15.3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
import os
import psutil
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score
from autogluon.tabular import TabularDataset, TabularPredictor

In [3]:
df = pd.read_csv("../../data/creditcard.csv")

In [4]:
# =========================
# COLUNAS
# =========================

X = df.drop(['Class'], axis=1)
y = df['Class'].astype(int)


In [5]:
# =========================
# FUNÇÃO AUXILIAR DE RECURSOS
# =========================
def get_memory_usage_mb():
    """Retorna o uso atual de memória RAM do processo em Megabytes."""
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 2)

# =========================
# INSTALAR AUTOGLUON
# =========================
# %pip install autogluon psutil

for SEED in [42, 101, 223, 302, 404]:
    print(f"\n================================")
    print(f"EXECUTANDO COM SEED: {SEED}")
    print(f"================================")

    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    
    # =========================
    # DATAFRAME FINAL
    # =========================
    train_df = pd.DataFrame(X_train)
    train_df['Class'] = y_train.values

    test_df = pd.DataFrame(X_test)
    test_df['Class'] = y_test.values

    train_data = TabularDataset(train_df)
    test_data = TabularDataset(test_df)

    # =========================
    # AUTOGLUON (TREINAMENTO + MÉTRICA COMPUTACIONAL)
    # =========================
    print("Iniciando treinamento do AutoGluon...")
    mem_antes_treino = get_memory_usage_mb()
    inicio_treino = time.perf_counter()
    
    predictor = TabularPredictor(
        label='Class',
        eval_metric='f1' # Prioriza o F1-score da classe minoritária
    ).fit(
        train_data=train_data,
        presets='medium_quality',
        hyperparameters={
            'GBM': {'class_weight': 'balanced'}, # Força pesos balanceados no LightGBM
            'XGB': {'scale_pos_weight': 587},      # Penaliza erros na classe minoritária -> 587 é a razão entre as classes majoritária e minoritária
            'CAT': {'auto_class_weights': 'Balanced'},
            'RF': {'class_weight': 'balanced'},
            'XT': {'class_weight': 'balanced'},
            "LR": {"class_weight": "balanced"},
        },
        time_limit=300
    )
    
    fim_treino = time.perf_counter()
    tempo_treino_total = round(fim_treino - inicio_treino, 2)
    mem_depois_treino = get_memory_usage_mb()
    memoria_treino_delta = round(mem_depois_treino - mem_antes_treino, 2)

    # =========================
    # LEADERBOARD
    # =========================
    print("\n===== LEADERBOARD =====")
    leaderboard = predictor.leaderboard(test_data, display=True)

    # =========================
    # MÉTRICAS DE TODOS OS MODELOS
    # =========================
    resultados = []

    X_test_ag = test_df.drop(columns=['Class'])
    y_true = test_df['Class'].values

    # Iterar sobre todos os modelos gerados
    for model_name in predictor.model_names():
        # Medição de tempo e uso de CPU/Memória na inferência específica do modelo
        inicio_inferencia = time.perf_counter()
        
        preds = predictor.predict(X_test_ag, model=model_name)
        y_pred = preds.astype(int).values
        
        y_score = predictor.predict_proba(X_test_ag, model=model_name, as_multiclass=False).values
        
        fim_inferencia = time.perf_counter()
        tempo_inferencia = round(fim_inferencia - inicio_inferencia, 4)

        # Métricas preditivas
        acc = round(accuracy_score(y_true, y_pred), 4)
        precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        f1_weighted = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)
        f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        average_precision = round(average_precision_score(y_true, y_score), 4)

        resultados.append({
            "Modelo": model_name,
            "Tipo": model_name.split('_')[0],
            "Seed": SEED,
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "F1-score Weighted": f1_weighted,
            "Average Precision": average_precision,
            "Tempo Treino Total (s)": tempo_treino_total,
            "Tempo Inferência (s)": tempo_inferencia,
            "Uso RAM Treino (MB)": mem_depois_treino
        })

    # =========================
    # RESULTADO FINAL
    # =========================
    df_resultados = pd.DataFrame(resultados)

    print("\n===== MÉTRiCAS E CUSTO COMPUTACIONAL - SEED {} =====".format(SEED))
    print(df_resultados.sort_values(by="Average Precision", ascending=False).to_string(index=False))

No path specified. Models will be saved in: "AutogluonModels/ag-20261108_214542"
Verbosity: 2 (Standard Logging)



EXECUTANDO COM SEED: 42
Iniciando treinamento do AutoGluon...


=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.90 GB / 16.00 GB (24.4%)
Disk Space Avail:   225.41 GB / 460.38 GB (49.0%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/lannaces/Documents/Inatel/TCC-autoML/notebooks/cost-sensitive/AutogluonModels/ag-20261108_214542"
Train Data Rows:    227845
Train Data Columns: 30
Label Column:       Class
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  [np.in


===== LEADERBOARD =====
                 model  score_test  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0           ExtraTrees    0.882979   0.857143          f1        0.160779       0.026247   3.549701                 0.160779                0.026247           3.549701            1       True          4
1         RandomForest    0.843243   0.750000          f1        0.074669       0.033084  19.710874                 0.074669                0.033084          19.710874            1       True          2
2             LightGBM    0.738318   0.857143          f1        0.003208       0.000480   0.679258                 0.003208                0.000480           0.679258            1       True          1
3  WeightedEnsemble_L2    0.738318   0.857143          f1        0.004041       0.001414   0.842395                 0.000833                0.000934           0.16

No path specified. Models will be saved in: "AutogluonModels/ag-20261108_214619"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.61 GB / 16.00 GB (22.6%)
Disk Space Avail:   225.28 GB / 460.38 GB (48.9%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/lannaces/Documents/Inatel/TCC-autoML/notebooks/cost-sensitive/AutogluonModels/ag-20261108_214619"
Train Data Rows:    227845
Train Data Columns: 30
Label Column:       Class
AutoGluon infers your


===== MÉTRiCAS E CUSTO COMPUTACIONAL - SEED 42 =====
             Modelo             Tipo  Seed  Accuracy  Precision  Recall  F1-score  F1-score Weighted  Average Precision  Tempo Treino Total (s)  Tempo Inferência (s)  Uso RAM Treino (MB)
         ExtraTrees       ExtraTrees    42    0.9996     0.9222  0.8469    0.8830             0.9996             0.8743                   35.61                0.2714              1076.28
       RandomForest     RandomForest    42    0.9995     0.8966  0.7959    0.8432             0.9995             0.8717                   35.61                0.1612              1076.28
            XGBoost          XGBoost    42    0.9976     0.4028  0.8673    0.5502             0.9980             0.8073                   35.61                0.0706              1076.28
           LightGBM         LightGBM    42    0.9990     0.6810  0.8061    0.7383             0.9991             0.7860                   35.61                0.0076              1076.28
WeightedEns

... (10 outputs left out)

No path specified. Models will be saved in: "AutogluonModels/ag-20261108_214819"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.72 GB / 16.00 GB (23.3%)
Disk Space Avail:   224.99 GB / 460.38 GB (48.9%)
Presets specified: ['medium_quality']



===== MÉTRiCAS E CUSTO COMPUTACIONAL - SEED 302 =====
             Modelo             Tipo  Seed  Accuracy  Precision  Recall  F1-score  F1-score Weighted  Average Precision  Tempo Treino Total (s)  Tempo Inferência (s)  Uso RAM Treino (MB)
            XGBoost          XGBoost   302    0.9993     0.7679  0.8776    0.8190             0.9994             0.8925                   38.38                0.1271              1592.88
       RandomForest     RandomForest   302    0.9996     0.9213  0.8367    0.8770             0.9996             0.8910                   38.38                0.2332              1592.88
         ExtraTrees       ExtraTrees   302    0.9996     0.9318  0.8367    0.8817             0.9996             0.8880                   38.38                0.4036              1592.88
           LightGBM         LightGBM   302    0.9994     0.8163  0.8163    0.8163             0.9994             0.8402                   38.38                0.0233              1592.88
WeightedEn

Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/lannaces/Documents/Inatel/TCC-autoML/notebooks/cost-sensitive/AutogluonModels/ag-20261108_214819"
Train Data Rows:    227845
Train Data Columns: 30
Label Column:       Class
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  [np.int64(0), np.int64(1)]
	If 'binary' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       binary
Preprocessing data...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    3834.84 MB
	Train Data (Original)  Memory Usage: 52.15 MB (1.4% of available memory)
	Inferring data type of each feature based 


===== LEADERBOARD =====
                 model  score_test  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0             CatBoost    0.843750   0.888889          f1        0.017375       0.012194  15.076659                 0.017375                0.012194          15.076659            1       True          3
1         RandomForest    0.829545   1.000000          f1        0.087982       0.040879  24.033074                 0.087982                0.040879          24.033074            1       True          2
2           ExtraTrees    0.829545   1.000000          f1        0.164028       0.049978   4.939836                 0.164028                0.049978           4.939836            1       True          4
3             LightGBM    0.791667   1.000000          f1        0.002866       0.000464   0.569899                 0.002866                0.000464           0.56